# (EC)DSA Q&A

## How are DSA and ElGamal signatures related?

ElGamal and DSA have very similar signing and verification operations. DSA signatures are (at least?) a performance improvement on ElGamal signature. I believe any computational differences between signing and verification are just a matter of DSA keeping modular exponentiations smaller. ElGamal signatures do modular exponentation with much larger exponents as they are elements of the group of integers modulo some large prime $p$. 

In constrast, DSA works in a subgroup that has a order much smaller than the size of $Z_p$ (e.g. $p$ can be 1024-bits, while the order of the subgroup is 160-bits). This allows exponentiation operations to be smaller.

[Thomas Pornin calls this out on this Stack Exchange post "DSA vs. ElGamal vs. Schnorr"](https://crypto.stackexchange.com/questions/10388/dsa-vs-elgamal-vs-schnorr/10389#10389).

## Why do DSA signatures use two distinct moduli?

During [DSA signing](https://en.wikipedia.org/wiki/Digital_Signature_Algorithm#Signing) the value $r$ is computed as: $$r = \left(g^{k} \mod p \right) \mod q$$

The two moduli is a bit odd. It is, however, a mapping from the larger group $Z_p$ to a different group $Z_q$ where $q$ is the order of the subgroup generated by $g$.

Why is this needed? Because the value $r$ (and $s$ too) are used as exponents of $g$ during verification. Given that the order of the subgroup generated by $g$ is $q$ any exponent larger than $q$ is unnecessary (That is, $g^q = 1$ so $g^{q+k} = g^{k}$). Mapping a element of $Z_p$ to $Z_q$ is just reducing larger-than-$q$ numbers to be smaller than $q$.

From what I gather, this mapping from $Z_p$ to $Z_q$ is one of the painpoints for proving the security of (EC)DSA. "On the Provable Security of (EC)DSA Signatures" covers this.

Relevant Stack Exchange link to this question: [DSA: why using mod p mod q and not just mod p](https://crypto.stackexchange.com/questions/68614/dsa-why-using-mod-p-mod-q-and-not-just-mod-p)


## Other References

https://security.stackexchange.com/questions/56892/are-dsa-keys-breakable

https://crypto.stackexchange.com/questions/51995/what-is-the-discrete-logarithm-problem-in-z-zp-in-the-dsa?rq=1

https://crypto.stackexchange.com/questions/13163/why-do-we-use-1024-160-bit-primes-in-dsa

In [28]:
# Parameters lifted from https://www.ietf.org/rfc/rfc6979.txt

# A 1024-bit prime
p = Integer('86F5CA03DCFEB225063FF830A0C769B9DD9D6153AD91D7CE27F787C43278B447'
            'E6533B86B18BED6E8A48B784A14C252C5BE0DBF60B86D6385BD2F12FB763ED88'
            '73ABFD3F5BA2E0A8C0A59082EAC056935E529DAF7C610467899C77ADEDFC846C'
            '881870B7B19B2B58F9BE0521A17002E3BDD6B86685EE90B3D9A1B02B782B1779', 16)

# Integers modulo p
Zp = Zmod(p)
# Integers modulo p-1
Zp1 = Zmod(p-1)

# Generator of a subgroup in Zp
g = Zp(Integer('07B0F92546150B62514BB771E2A0C0CE387F03BDA6C56B505209FF25FD3C133D'
               '89BBCD97E904E09114D9A7DEFDEADFC9078EA544D2E401AEECC40BB9FBBF78FD'
               '87995A10A1C27CB7789B594BA7EFB5C4326A9FE59A070E136DB77175464ADCA4'
               '17BE5DCE2F40D10A46A3A3943F26AB7FD9C0398FF8C76EE0A56826A8A88F1DBD', 16))

# The order of the subgroup generated by g. 160-bits.
q = Integer('996F967F6C8E388D9E28D01E205FBA957A5698B1', 16)

# Quick test that q appears to be the order of subgroup generated by g.
assert g^q == 1